# CSED 504 — A2 NLP: Hello Text

Sanity-check notebook for the natural language project.  It:

1. Verifies the environment (GPU detection, package versions)
2. Sets up a character-level tokenizer on a small text corpus
3. Builds a **GPT-style causal language model** from scratch using pure PyTorch
4. Runs a forward pass, verifies output shape, and generates a few characters

If this runs end-to-end without errors, the environment is ready for A2.

---
**To use in Google Colab:**
```
!git clone https://github.com/TrueRottweiler/WashingtonCsed504.git
%cd WashingtonCsed504/src/a2-nlp
```
Then run the cells below.

In [ ]:
# Install required packages.  If you're running in the uw-csed504 conda environment these
# are already installed and this is a no-op.  For Colab or fresh setups it will pull them in
# automatically.
%pip install --quiet torch numpy matplotlib tqdm

In [ ]:
# -- 1. Path setup ---------------------------------------------------------------------------------
# Add the shared common/ utility folder to Python's search path so we can import gpu_check.py
# from there.  Inserting at index 0 means it takes precedence over any installed packages with
# the same name.

import os, sys

_common = os.path.normpath(os.path.join(os.getcwd(), '../common'))
if os.path.isdir(_common) and _common not in sys.path:
    sys.path.insert(0, _common)

# -- 2. Device detection ---------------------------------------------------------------------------
# Transformers rely heavily on matrix multiplications.  A GPU runs those thousands of times faster
# than a CPU because it has thousands of small cores optimized for exactly this kind of parallel
# number-crunching.  We check for hardware in priority order:
#   1. CUDA  — NVIDIA GPU; the standard for ML, fastest overall
#   2. MPS   — Apple Silicon 'Metal Performance Shaders' (M1/M2/M3 Mac)
#   3. CPU   — always available; fine for this small model

try:
    from gpu_check import get_device, set_seed
    DEVICE = get_device()   # prints which device was found
    set_seed(42)            # lock all random seeds for reproducible results
except ImportError:
    # Fallback inline detection — runs if common/ isn't reachable
    import torch
    if torch.cuda.is_available():
        DEVICE = torch.device('cuda')
        _name  = torch.cuda.get_device_name(DEVICE)
        _gb    = torch.cuda.get_device_properties(DEVICE).total_memory / 1e9
        print(f'  Device   : CUDA — {_name} ({_gb:.1f} GB)')
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        try:
            torch.zeros(1, device='mps')
            DEVICE = torch.device('mps')
            print('  Device   : MPS — Apple Silicon GPU (unified memory)')
        except Exception:
            DEVICE = torch.device('cpu')
            print(f'  Device   : CPU ({os.cpu_count() or 1} logical cores)')
    else:
        DEVICE = torch.device('cpu')
        print(f'  Device   : CPU ({os.cpu_count() or 1} logical cores)')
    import random; import numpy as np
    random.seed(42); np.random.seed(42); torch.manual_seed(42)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

## 1. Sample Text

We use a character-level tokenizer — the simplest possible tokenizer — so there are no external
downloads or vocab files needed.  Each character maps to an integer index.  Real assignments will
use sub-word tokenizers (BPE, WordPiece) instead.

In [ ]:
# A tiny corpus to define the vocabulary and generate sample sequences from.
# We use Shakespeare because it's short, public-domain, and rich in vocabulary.
CORPUS = (
    'To be, or not to be, that is the question: '
    'Whether tis nobler in the mind to suffer '
    'The slings and arrows of outrageous fortune, '
    'Or to take arms against a sea of troubles.'
)

# -- Build character-level vocabulary --------------------------------------------------------------
# set(CORPUS) finds every unique character in the text.  sorted() makes the ordering deterministic
# — same vocab on every run.  enumerate() assigns a unique integer to each character.
vocab      = sorted(set(CORPUS))
VOCAB_SIZE = len(vocab)
stoi       = {ch: i for i, ch in enumerate(vocab)}   # char → index ('s to i')
itos       = {i: ch for ch, i in stoi.items()}        # index → char ('i to s')

# encode converts a string to a list of integers (for feeding into the model)
# decode converts a list of integers back to a readable string
encode = lambda s: [stoi[c] for c in s]
decode = lambda t: ''.join(itos[i] for i in t)

print(f'Corpus length : {len(CORPUS)} characters')
print(f'Vocabulary    : {VOCAB_SIZE} unique characters')
print(f'Vocab (first 20): {vocab[:20]}')

# -- Create (input, target) sequence pairs from the corpus -----------------------------------------
# SEQ_LEN is the 'context window' — how many characters the model sees at once.
SEQ_LEN    = 32
BATCH_SIZE = 8

tokens = torch.tensor(encode(CORPUS), dtype=torch.long)

# Pick BATCH_SIZE random starting positions in the token stream.
starts = torch.randint(0, len(tokens) - SEQ_LEN - 1, (BATCH_SIZE,))

# x_batch: the INPUT tokens the model reads
# y_batch: the TARGET tokens — y[t] is the character that should follow x[t]
#
# This is the 'next-token prediction' objective: given the first T characters, predict each of
# the next characters.  y_batch is just x_batch shifted one position to the right.  The model
# sees 'T, o, space' and must predict 'o, space, b', etc.  This is how GPT-style models train.
x_batch = torch.stack([tokens[s     : s + SEQ_LEN    ] for s in starts])
y_batch = torch.stack([tokens[s + 1 : s + SEQ_LEN + 1] for s in starts])

print(f'\nInput batch shape : {tuple(x_batch.shape)}')
print(f'Sample sequence   : {repr(decode(x_batch[0].tolist()))}')

## 2. Causal Language Model from Scratch

This is a scaled-down GPT: a **decoder-only transformer** that predicts the next character at every
position.  The key difference from the ViT encoder above is the **causal mask** — each token can
only attend to tokens that came before it, which is what makes autoregressive generation possible.

```
token indices (B, T)
  └─ token embedding + position embedding → (B, T, D)
       └─ × L TransformerBlocks (causal)  → (B, T, D)
          └─ LayerNorm → LM head          → (B, T, vocab_size)  logits
```

Reference: *Language Models are Unsupervised Multitask Learners* (Radford et al., 2019)
and Andrej Karpathy's nanoGPT.

In [ ]:
# Hyperparameters — nano scale, fast to run on any hardware
#
# EMBED_DIM: each character is represented as a vector of this length.  Larger → richer
#   representations, but more parameters to train.  MUST be divisible by NUM_HEADS.
#
# NUM_HEADS: attention is split into independent 'heads', each operating on an
#   EMBED_DIM/NUM_HEADS slice.  Different heads learn different patterns (e.g., one head might
#   track verb-noun agreement, another handles punctuation).
#
# NUM_LAYERS: depth of the transformer.  Each layer lets every token attend to all prior tokens
#   and update its representation.  Deeper → more expressive.
#
# FFN_DIM: the hidden dimension of the MLP inside each block.  Convention is 4× EMBED_DIM;
#   we use 2× here to keep the model fast.
#
# DROPOUT: random zeroing of activations during training prevents the model from memorizing
#   the training data instead of learning general patterns.
EMBED_DIM  = 128
NUM_HEADS  = 4     # 128 / 4 = 32 dimensions per head
NUM_LAYERS = 4
FFN_DIM    = 256   # 2× EMBED_DIM
DROPOUT    = 0.1

In [ ]:
class CausalSelfAttention(nn.Module):
    """
    Masked multi-head self-attention — the 'decoder-style' attention block.

    The causal (lower-triangular) mask ensures position i can only attend to positions 0..i, never
    to the future.  This is what makes the model autoregressive: generation is always left-to-right.

    We implement scaled dot-product attention by hand here rather than
    delegating to nn.MultiheadAttention so the mechanics are explicit.
    (PyTorch 2.0+ has F.scaled_dot_product_attention with FlashAttention,
    but the manual version is easier to learn from.)
    """

    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        assert embed_dim % num_heads == 0, 'embed_dim must be divisible by num_heads'
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads

        # Why scale by 1/sqrt(head_dim)?  Q @ K^T produces dot products whose magnitude grows
        # with head_dim.  Large values push softmax into its saturated 'almost-one-hot' region
        # where gradients nearly vanish.  Dividing by sqrt(head_dim) keeps variance ≈ 1.
        self.scale = self.head_dim ** -0.5

        # One fused linear layer computes Q, K, and V simultaneously.  Output is 3× embed_dim
        # so we can split it into three equal parts.  bias=False is a common convention.
        self.qkv  = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        # Final projection mixes information across the heads after attention
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, D = x.shape
        H, Dh   = self.num_heads, self.head_dim

        # -- Step 1: Compute Q (query), K (key), V (value) -----------------------------------------
        # One matrix multiply produces all three projections at once.
        # Intuition: Q = 'what am I looking for?'  K = 'what do I contain?'
        # V = 'what information do I pass on if selected?'
        qkv     = self.qkv(x)               # (B, T, 3D)
        Q, K, V = qkv.chunk(3, dim=-1)      # each (B, T, D)

        # Reshape so each head operates on its own Dh-dimensional slice.  .view() reshapes
        # without copying; .transpose(1,2) swaps heads before time.  Think of it as H
        # independent attention problems running in parallel.
        Q = Q.view(B, T, H, Dh).transpose(1, 2)   # (B, H, T, Dh)
        K = K.view(B, T, H, Dh).transpose(1, 2)
        V = V.view(B, T, H, Dh).transpose(1, 2)

        # -- Step 2: Scaled dot-product attention scores -------------------------------------------
        # A[b, h, i, j] = 'how much should position i attend to position j?'
        # Scaling by 1/sqrt(Dh) keeps variance stable regardless of head size.
        A = (Q @ K.transpose(-2, -1)) * self.scale   # (B, H, T, T)

        # -- Step 3: Apply the causal mask ---------------------------------------------------------
        # torch.tril gives a lower-triangular matrix (True where i >= j).  We fill the UPPER
        # triangle (future positions) with -inf so softmax maps them to exactly 0 — the model
        # can't cheat by peeking at token t+1 while learning to predict it.
        mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
        A    = A.masked_fill(~mask, float('-inf'))
        A    = F.softmax(A, dim=-1)   # normalize rows to sum to 1
        A    = self.drop(A)

        # -- Step 4: Weighted sum over values, then merge heads ------------------------------------
        # Each position gets a weighted average of the value vectors it is allowed to attend to.
        out = A @ V                                          # (B, H, T, Dh)
        # .contiguous() ensures packed memory layout before .view()
        out = out.transpose(1, 2).contiguous().view(B, T, D)    # (B, T, D)
        # The final projection mixes information across the different heads
        return self.proj(out)

In [ ]:
class TransformerBlock(nn.Module):
    """
    GPT-style transformer block: Pre-LayerNorm, causal self-attention, FFN.

    Same structure as the ViT block in hello_image.ipynb except:
      • attention is CausalSelfAttention (decoder-style) not bidirectional
      • implemented manually so the attention mechanics are visible
    """

    def __init__(self, embed_dim, num_heads, ffn_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = CausalSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)

        # Attention tells each token WHAT to look at; the FFN tells it WHAT TO DO with that
        # information.  Both are needed: attention alone has no nonlinearity to model complex
        # patterns, and FFN alone can't mix context.  GELU (smoother than ReLU) works better
        # empirically in transformers.
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # Both sub-layers use RESIDUAL connections: output = input + transform(norm(input)).
        # Why residuals?  They let gradients flow straight back through the skip path during
        # training, preventing the vanishing-gradient problem that plagued earlier deep networks.
        x = x + self.attn(self.norm1(x))   # attention with skip connection
        x = x + self.ffn(self.norm2(x))    # FFN with skip connection
        return x

In [ ]:
class GPT(nn.Module):
    """
    Nano-scale GPT — a causal character-level language model.

    Weight tying: the token embedding matrix is shared with the output LM head.  This halves that
    portion of the parameter count and generally improves perplexity — it's a standard trick from
    Press & Wolf (2016) and used in the original GPT-2 paper.
    """

    def __init__(self, vocab_size, seq_len, embed_dim,
                 num_heads, num_layers, ffn_dim, dropout=0.1):
        super().__init__()
        self.seq_len = seq_len

        # Token embedding: a lookup table mapping each character index to a learned vector of size
        # embed_dim.  This is how the model 'reads' text — each integer becomes a dense vector.
        self.tok_embed = nn.Embedding(vocab_size, embed_dim)

        # Position embedding: a lookup table with one row per sequence position.  WHY is this
        # needed?  Self-attention is PERMUTATION INVARIANT — it treats the input as an unordered
        # set.  Swapping tokens 3 and 7 would give the exact same output without this.  Adding a
        # distinct vector for each position teaches the model 'this is character 5 in the window'.
        self.pos_embed = nn.Embedding(seq_len, embed_dim)
        self.drop      = nn.Dropout(dropout)

        self.blocks = nn.Sequential(*[
            TransformerBlock(embed_dim, num_heads, ffn_dim, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(embed_dim)

        # Language modeling head: projects from embed_dim to vocab_size.  Output is a score
        # (logit) for each possible next character.
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)

        # Weight tying: make the LM head use the SAME weight matrix as the input embedding.
        # Intuition: if 'the letter A' has vector v in the input, it makes sense to score
        # 'output is A' using that same v.  This also halves that matrix's parameter count.
        self.head.weight = self.tok_embed.weight

    def forward(self, idx):
        """idx: (B, T) integer token indices.  Returns (B, T, vocab_size) logits."""
        B, T = idx.shape
        assert T <= self.seq_len, f'Sequence length {T} > context window {self.seq_len}'

        # Create position indices [0, 1, 2, ..., T-1] — one integer per token.
        positions = torch.arange(T, device=idx.device)

        # ADD token embedding + position embedding (elementwise).  Each resulting vector encodes
        # both WHAT the character is (content) and WHERE it appears in the window (position).
        x = self.drop(self.tok_embed(idx) + self.pos_embed(positions))   # (B, T, D)

        # Run through all transformer blocks.  Each block lets every token attend to the tokens
        # before it and update its representation.
        x = self.blocks(x)
        x = self.norm(x)
        return self.head(x)                                                # (B, T, vocab_size)

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        """
        Autoregressively sample max_new_tokens tokens given a prompt.
        idx: (B, T) starting token sequence.
        temperature > 1 → more random/creative;  < 1 → more conservative/repetitive.
        """
        for _ in range(max_new_tokens):
            # Crop the context to seq_len if it has grown too long.
            # The model can only attend to at most seq_len tokens at a time.
            idx_cond = idx[:, -self.seq_len:]

            # Forward pass: get logits for every position
            logits = self(idx_cond)

            # We only care about the LAST position — the prediction for the NEXT token.
            # Temperature rescales the logits before softmax: high temp flattens the distribution
            # (more random); low temp sharpens it (model picks its top choice almost every time).
            logits   = logits[:, -1, :] / temperature
            probs    = F.softmax(logits, dim=-1)

            # Sample one token from the probability distribution and append it.
            # torch.multinomial draws a sample proportional to the probabilities.
            next_tok = torch.multinomial(probs, num_samples=1)   # (B, 1)
            idx      = torch.cat([idx, next_tok], dim=1)
        return idx

    def num_parameters(self):
        # Count unique parameters — weight tying means tok_embed and head share one tensor, so we
        # de-dup by id() to avoid double-counting.
        seen, total = set(), 0
        for p in self.parameters():
            if id(p) not in seen and p.requires_grad:
                seen.add(id(p))
                total += p.numel()
        return total

## 3. Smoke Test

Instantiate the model, run a forward pass, verify the output shape, then generate a short sequence
from a prompt.  The generated text will be gibberish at this point — the model is untrained — but
the shapes and mechanics should all be correct.

In [ ]:
# Build the model and move all parameters to the detected device.
model = GPT(
    vocab_size = VOCAB_SIZE,
    seq_len    = SEQ_LEN,
    embed_dim  = EMBED_DIM,
    num_heads  = NUM_HEADS,
    num_layers = NUM_LAYERS,
    ffn_dim    = FFN_DIM,
    dropout    = DROPOUT,
).to(DEVICE)

print(f'Model parameters : {model.num_parameters():,}')
print(f'Vocab size       : {VOCAB_SIZE}')
print(f'Context window   : {SEQ_LEN}')

# -- Forward pass ----------------------------------------------------------------------------------
# eval() turns off dropout so every run gives the same deterministic output.
# no_grad() skips building the backprop graph — halves memory and speeds up inference.
model.eval()
with torch.no_grad():
    x      = x_batch.to(DEVICE)
    logits = model(x)               # (B, T, vocab_size): one score per char per position

print(f'\nInput  shape     : {tuple(x.shape)}')
expected = f'({BATCH_SIZE}, {SEQ_LEN}, {VOCAB_SIZE})'
print(f'Output shape     : {tuple(logits.shape)}  — expected {expected}')

assert logits.shape == (BATCH_SIZE, SEQ_LEN, VOCAB_SIZE), 'Output shape mismatch!'

# -- Generation smoke test -------------------------------------------------------------------------
# Encode a short prompt, generate 40 characters autoregressively, decode and print.  The output is
# random nonsense (the model is untrained) but proves the forward pass, sampling, and decode
# pipeline all work end-to-end.
PROMPT = 'To be'
prompt_ids = torch.tensor([encode(PROMPT)], dtype=torch.long).to(DEVICE)

with torch.no_grad():
    generated = model.generate(prompt_ids, max_new_tokens=40, temperature=1.0)

generated_text = decode(generated[0].tolist())
print(f'\nPrompt    : {repr(PROMPT)}')
print(f'Generated : {repr(generated_text)}')
print()
print('All checks passed.  Environment is ready for A2.')